# Big Data Project — Medallion Pipeline: chess_games

Bronze / Silver / Gold Pipeline auf Databricks mit Quality Assessment gegen die
clean-Referenz. Datensatz: **chess_games** (ZHAW Benchmark).

**So ist das Notebook aufgebaut:** Die Abschnitte Bronze / Silver / Gold sind
klar getrennt. Die Infrastruktur (Ingest, Shadow-Columns, 5-Kategorien-Logik,
KPIs, Iteration-Log) ist fertig. Die **Cleansing-Logik pro Feld** ist als Stub
hinterlegt — das ist eure eigentliche Aufgabe. Lasst das Notebook einmal
komplett laufen: es funktioniert sofort end-to-end und liefert eine
Baseline-KPI-Zeile (alle Fehler = FN, Recall 0). Das ist die erste Zeile eures
Iteration-Logs. Danach füllt ihr die Stubs und iteriert.

## Feldauswahl und Vorüberlegungen

Ein Vergleich beider CSVs zeigt **5 Spalten mit echten Fehlern**. `rated`,
`created_at` und `last_move_at` zeigen zwar Differenzen, sind aber reine
CSV-Export-Artefakte (Gross-/Kleinschreibung bzw. Zahlenformat) — **keine**
Fehlerfelder. Alle übrigen Spalten sind sauber.

| Feld | Fehlertyp | Anzahl |
|---|---|---|
| `opening_name` | Typo | 3080 |
| `winner` | Missing Value | 1995 |
| `victory_status` | Illegal Value | 1994 |
| `black_rating` | Illegal/Wrong Value | 1484 |
| `white_rating` | Illegal/Wrong Value | 1446 |

**Analysierte Felder (3 Personen → 4 Felder):** `winner`, `victory_status`,
`white_rating`, `opening_name`. `black_rating` ist optionaler Bonus (in
`ANALYSED_FIELDS` unten eine Zeile ergänzen).

**Row-ID:** `chess_games` hat zwar eine `id`-Spalte, diese ist aber **nicht
eindeutig** (945 Duplikate von 20'058 Zeilen). Ein Join auf `id` würde in Gold
einen Fan-out erzeugen. Beide Dateien haben identische Zeilenreihenfolge —
daher vergeben wir in Bronze einen positionsbasierten `row_id` und joinen
darauf.

**Hinweis zur KI-Nutzung:** Dieses
Pipeline-Gerüst — Bronze/Silver/Gold-Struktur, Shadow-Column-Mechanik,
5-Kategorien-Logik, KPI-Berechnung, Iteration-Log sowie die `winner`-Cleansing
als Referenzbeispiel — wurde mit KI-Unterstützung erstellt. Die Feldauswahl
wurde durch eigene Datenanalyse validiert. Die Cleansing-Logik für
`victory_status`, `white_rating` und `opening_name` ist vom Team zu
implementieren und zu validieren.

In [0]:
# === Setup ===
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()   # auf Databricks: bestehende Session

# --- ANPASSEN: Pfade zu euren hochgeladenen CSVs ---------------------------
# Je nachdem, wohin ihr in Databricks hochgeladen habt, z.B.:
#   Volume:     /Volumes/<catalog>/<schema>/<volume>/chess_games.csv
CLEAN_PATH = "/Volumes/workspace/default/chess/chess_games.csv"
DIRTY_PATH = "/Volumes/workspace/default/chess/dirty_chess_games.csv"

SCHEMA = "chess_project"
ANALYSED_FIELDS = ["winner", "victory_status", "white_rating", "opening_name"]
# black_rating als Bonus: einfach hier ergänzen und in cast_types casten.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE {SCHEMA}")

def save_delta(df, name):
    """Schreibt df als Delta-Tabelle (Medallion-Layer) und gibt sie zurück."""
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(name))
    return spark.table(name)

## BRONZE — Roh-Ingest

Beide CSVs werden vollständig geladen, alle Spalten als String (rohester
Zustand, keine Typkonvertierung). Jede Zeile bekommt einen stabilen `row_id`
plus Ingest-Metadaten. Bronze ist die Single Source of Truth.

In [0]:
# === BRONZE ===
def ingest_bronze(path, source):
    df = spark.read.option("header", True).csv(path)   # alle Spalten als String
    # row_id: positionsbasiert. coalesce(1) ist ZWINGEND, damit die IDs
    # zwischen clean und dirty zeilenweise übereinstimmen. monotonically_
    # increasing_id() liefert in einer einzelnen Partition 0,1,2,... in
    # Dateireihenfolge.
    df = df.coalesce(1).withColumn("row_id", F.monotonically_increasing_id())
    return (df.withColumn("_source", F.lit(source))
              .withColumn("_ingest_ts", F.current_timestamp()))

clean_bronze = save_delta(ingest_bronze(CLEAN_PATH, "clean"), "clean_bronze")
dirty_bronze = save_delta(ingest_bronze(DIRTY_PATH, "dirty"), "dirty_bronze")

print("Bronze | clean:", clean_bronze.count(), " dirty:", dirty_bronze.count())
clean_bronze.select("row_id", "id", *ANALYSED_FIELDS).show(5, truncate=False)

## SILVER — typisiert und bereinigt

`clean_silver`: nur Typkonvertierung, keine Cleansing-Regeln — die clean-Daten
sind die Referenz, kein Korrekturkandidat.

`dirty_silver`: Typkonvertierung + eure Cleansing-Logik. Für **jedes**
analysierte Feld `X` legen wir eine **Shadow-Column** `X_original` an: der Wert
*vor* der Bereinigung. Der bereinigte Wert lebt in `X` selbst. Daraus leitet
Gold ab, ob euer System eine Korrektur vorgenommen hat (`X != X_original`).

In [0]:
# === SILVER: Typkonvertierung ===
def cast_types(df):
    """Nur Typkonvertierung. Gilt für clean UND dirty identisch, damit der
    Vergleich in Gold apples-to-apples ist."""
    return (df.withColumn("white_rating", F.col("white_rating").cast("int"))
              .withColumn("black_rating", F.col("black_rating").cast("int"))
              .withColumn("turns",        F.col("turns").cast("int")))

clean_silver = save_delta(cast_types(clean_bronze), "clean_silver")
print("clean_silver bereit:", clean_silver.count(), "Zeilen")

### Cleansing-Funktionen

Konvention: jede Funktion bekommt ein DataFrame und gibt es mit
**überschriebener** Spalte `X` zurück. Die Shadow-Column `X_original` wird
zentral vorher angelegt.



In [0]:
# === SILVER: Cleansing-Funktionen ===

def clean_winner(df):
    """winner — Missing Value.  [REFERENZBEISPIEL, KI-unterstuetzt erstellt]

    ERKENNUNG: winner ist leer/null -> fehlender Wert.
    REPARATUR: Endet der letzte Zug in `moves` auf '#' (Schachmatt), hat die
      Seite gewonnen, die den letzten Zug gemacht hat. `turns` = Anzahl Halb-
      zuege: ungerade -> Weiss zog zuletzt, gerade -> Schwarz.
    GRENZE: Bei Aufgabe/Zeitueberschreitung ist der Sieger aus den Daten nicht
      ableitbar -> diese Fälle bleiben offen (zählen als FN). Genau diese
      Teil-Reparierbarkeit gehört in den Report.
    """
    is_missing = F.col("winner").isNull() | (F.trim(F.col("winner")) == "")
    inferred = F.when(F.col("turns") % 2 == 1, F.lit("white")).otherwise(F.lit("black"))
    repaired = F.when(is_missing & F.col("moves").endswith("#"), inferred) \
                .otherwise(F.col("winner"))
    return df.withColumn("winner", repaired)


def clean_victory_status(df):
    """victory_status — Illegal Value.  [STUB — Person C]

    ERKENNUNG: gültige Werte = {mate, resign, outoftime, draw}. Alles andere
      (in den dirty-Daten: 'regicide') ist ein illegaler Wert.
    REPARATUR (Vorschläge): moves endet auf '#' -> 'mate'; bereinigter winner
      == 'draw' -> 'draw'. resign vs outoftime ist NICHT unterscheidbar ->
      offen lassen und in Abschnitt 9 als unverifizierbar dokumentieren.
    """
    return df  # Baseline: identity


def clean_white_rating(df):
    """white_rating — Illegal/Wrong Value.  [STUB — Person C]

    ERKENNUNG: Wert ausserhalb eines plausiblen Elo-Bereichs (Fehlerwerte sind
      durchweg 5-stellig; gueltige Ratings ~700-2800). Schwelle im Report
      begruenden.
    REPARATUR: Originalwert ist aus den Daten nicht rekonstruierbar. Auf null
      setzen (-> TP-miss) oder grob schätzen. Erwartung: Recall ~100%, Repair
      Accuracy ~0% — Kernbeispiel fuer Abschnitt 9.
    """
    return df  # Baseline: identity


def clean_opening_name(df):
    """opening_name — Typo.  [STUB — Person B]

    ERKENNUNG: Name nicht im Wörterbuch gueltiger Eröffnungsnamen. Wörter-
      buch aus den häufigen Namen im dirty-Stream selbst bauen.
    REPARATUR: Fuzzy-Matching (rapidfuzz/Levenshtein) gegen das Wörterbuch;
      opening_eco (sauber) grenzt die Kandidaten ein. In Spark: Wörterbuch
      broadcasten + UDF, oder pandas_udf.
    """
    return df  # Baseline: identity

In [0]:
# === SILVER: dirty_silver bauen ===
def build_dirty_silver(df):
    df = cast_types(df)
    for fld in ANALYSED_FIELDS:                       # Shadow-Columns ZUERST
        df = df.withColumn(f"{fld}_original", F.col(fld))
    df = clean_winner(df)                             # dann Cleansing anwenden
    df = clean_victory_status(df)
    df = clean_white_rating(df)
    df = clean_opening_name(df)
    return df

dirty_silver = save_delta(build_dirty_silver(dirty_bronze), "dirty_silver")
print("dirty_silver bereit. Spalten je Feld: X und X_original")
dirty_silver.select("row_id", "winner", "winner_original").show(5)

## GOLD — Business-Layer und Quality-Messung

Inner Join von `clean_silver` und `dirty_silver` über `row_id`. Für jede
analysierte Zelle wird die 5-Kategorie bestimmt:

| Ground Truth | euer System | Match clean? | Kategorie |
|---|---|---|---|
| Fehler | korrigiert | ja | TP-hit |
| Fehler | korrigiert | nein | TP-miss |
| Fehler | unverändert | – | FN |
| kein Fehler | unverändert | ja | TN |
| kein Fehler | korrigiert | nein | FP |

In [0]:
# === GOLD: Join + 5-Kategorien-Klassifikation ===
clean_sel = clean_silver.select(
    "row_id", *[F.col(f).alias(f"{f}_clean") for f in ANALYSED_FIELDS])
gold = dirty_silver.join(clean_sel, on="row_id", how="inner")
print("Gold: gejointe Zeilen =", gold.count())

def categorize(df, field):
    """Ordnet jede Zelle des Feldes genau einer der 5 Kategorien zu.
    eqNullSafe behandelt null == null korrekt (wichtig, falls Cleansing
    Werte auf null setzt)."""
    x       = F.col(field)                  # bereinigter dirty-Wert
    x_orig  = F.col(f"{field}_original")    # dirty-Wert vor Cleansing
    x_clean = F.col(f"{field}_clean")       # clean-Referenz
    error_present = ~x_orig.eqNullSafe(x_clean)
    corrected     = ~x.eqNullSafe(x_orig)
    matches_clean =  x.eqNullSafe(x_clean)
    cat = (F.when(error_present & corrected & matches_clean,  "TP_hit")
            .when(error_present & corrected & ~matches_clean, "TP_miss")
            .when(error_present & ~corrected,                 "FN")
            .when(~error_present & ~corrected,                "TN")
            .when(~error_present & corrected,                 "FP")
            .otherwise("UNDEF"))
    return df.select("row_id", F.lit(field).alias("field"), cat.alias("category"))

categories = None
for f in ANALYSED_FIELDS:
    c = categorize(gold, f)
    categories = c if categories is None else categories.unionByName(c)

categories = save_delta(categories, "gold_cell_categories")
categories.groupBy("field", "category").count().orderBy("field", "category").show()

In [0]:
# === GOLD: KPIs pro Feld + Aggregat ===
CAT_COLS = ["TP_hit", "TP_miss", "FN", "TN", "FP", "UNDEF"]

pivot = (categories.groupBy("field")
         .pivot("category", CAT_COLS).count().fillna(0))

# Aggregat-Zeile über alle Felder
agg = (pivot.agg(*[F.sum(c).alias(c) for c in CAT_COLS])
            .withColumn("field", F.lit("AGGREGATE")))
all_counts = pivot.unionByName(agg.select(pivot.columns))

# KPIs (mit sicherer Division)
tp = F.col("TP_hit") + F.col("TP_miss")
prec = F.when((tp + F.col("FP")) > 0, tp / (tp + F.col("FP")))
rec  = F.when((tp + F.col("FN")) > 0, tp / (tp + F.col("FN")))
kpis = (all_counts
        .withColumn("Precision", F.round(prec, 3))
        .withColumn("Recall",    F.round(rec, 3))
        .withColumn("F1",        F.round(2 * prec * rec / (prec + rec), 3))
        .withColumn("RepairAcc", F.round(F.when(tp > 0, F.col("TP_hit") / tp), 3))
        .withColumn("FP_Rate",   F.round(
            F.when((F.col("FP") + F.col("TN")) > 0,
                   F.col("FP") / (F.col("FP") + F.col("TN"))), 3)))

kpis = save_delta(kpis, "gold_kpis")
kpis.orderBy("field").show(truncate=False)
# Auf Databricks für eine schönere Tabelle:  display(kpis.orderBy("field"))

undef = kpis.agg(F.sum("UNDEF")).collect()[0][0]
print("UNDEF gesamt (muss 0 sein):", undef)

## Iteration-Log

Nach jeder Verfeinerung eurer Cleansing-Regeln einmal `log_iteration("...")`
aufrufen. Es schreibt die aktuellen Aggregat-KPIs mit Notiz und Zeitstempel in
die Tabelle `iteration_log` (Append). So entsteht die Entwicklung eurer
KPIs über die Iterationen — Pflichtbestandteil der Abgabe.

Zum Zurücksetzen: `spark.sql("DROP TABLE IF EXISTS iteration_log")`.

In [0]:
# === Iteration-Log ===
def log_iteration(note):
    """Schreibt die aktuellen Aggregat-KPIs als neue Iteration in iteration_log."""
    row = (kpis.filter(F.col("field") == "AGGREGATE")
               .withColumn("note", F.lit(note))
               .withColumn("logged_at", F.current_timestamp()))
    row.write.format("delta").mode("append").saveAsTable("iteration_log")
    print(f"Iteration protokolliert: {note}")

# Erste Iteration = Baseline (Stubs noch identity)
log_iteration("Iteration 0 - Baseline, nur winner implementiert")

spark.table("iteration_log").select(
    "logged_at", "note", "Recall", "Precision", "F1", "RepairAcc", "FP_Rate"
).orderBy("logged_at").show(truncate=False)

## Nächste Schritte / Aufgabenteilung

**Person A — Pipeline & Infrastruktur:** dieses Gerüst betreuen, `winner`
verfeinern (z.B. Remis-Fälle über `victory_status` ergänzen), Gold und
Iteration-Log pflegen, Report zusammenführen.

**Person B — `opening_name`:** `clean_opening_name` implementieren (Wörterbuch
+ Fuzzy-Matching mit rapidfuzz, eingegrenzt über `opening_eco`).

**Person C — `victory_status` + `white_rating`:** beide Stubs implementieren;
`white_rating` liefert das zentrale Beispiel für unverifizierbare Fülle.

Jede Person dokumentiert ihren Teil im Iteration-Log und im Report (Cleansing-
Ansatz + "nicht lösbar"-Diskussion für ihr Feld).

**Workflow pro Feld:** Stub implementieren → Notebook ab Silver neu ausführen →
KPIs in Gold ansehen → `log_iteration("...")` → Regel verfeinern → wiederholen.